In [20]:
import os
import glob
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from tabulate import tabulate
import matplotlib.pyplot as plt
from tabulate import tabulate

In [21]:
read_path = 'nlst780.idc.delivery.052821\\data\\'

## Read data

In [22]:
all_participants = pd.read_csv(read_path + 'participant_d040722.csv')
df_ctab = pd.read_csv(read_path + 'nlst_780_ctab_idc_20210527.csv')
df_ctabc = pd.read_csv(read_path + 'nlst_780_ctabc_idc_20210527.csv')

C:\Users\YiJin\AppData\Local\Temp\ipykernel_46056\4097394357.py:1: DtypeWarning: Columns (239,240,348) have mixed types. Specify dtype option on import or set low_memory=False.
  all_participants = pd.read_csv(read_path + 'participant_d040722.csv')


## Config

In [23]:
######################################################## COLMN NAMES ##################################################
PATIENT_ID_COLMN_NAME                      = 'pid'
PATIENT_GENDER_COLMN_NAME                  = 'gender'
PATIENT_AGE_COLMN_NAME                     = 'age'
PATIENT_RACE_COLMN_NAME                    = 'race'
PATIENT_STUDYYEAR_COLMN_NAME               ='study_yr'
PATIENT_ABNORNALITY_DESCRIPTION_COLMN_NAME = 'sct_ab_desc'



CANCER_GRP_COLMM_NAME                       = 'scr_group'
CANCER_SCR_COLMM_NAME                       = 'can_scr'
CANCER_RPT_LINK_COLMM_NAME                  = 'canc_rpt_link' # Is the diagnosis of lung cancer associated with a positive screen?
ANY_REPORTED_CANCER_COLMM_NAME              = 'anyscr_has_nodule'
LUNG_CANCER_COLMM_NAME                      = 'lung_cancer'                                                   
NODULE_LOC_COLMM_NAME                       = "sct_epi_loc"
NODULE_MARGIN_COLMM_NAME                    = "sct_margins"
NODULE_ATT_COLMM_NAME                       = "sct_pre_att"


ABNORNALITIES_TO_CONDIDER = [51]

#####################################################--Decoder---########################################################

GENDER_DECODER    = {1 :"male",2 :"female"}

ABNORNALITY_DESCRIPTION_DECODER = {51 : "Non-calcified nodule or mass (opacity >= 4 mm diameter)",
                      52 :"Non-calcified micronodule(s) (opacity < 4 mm diameter)",
                      53 :"Benign lung nodule(s) (benign calcification)",
                      54 :"Atelectasis, segmental or greater",
                      55 :"Pleural thickening or effusion",
                      56 :"Non-calcified hilar/mediastinal adenopathy or mass (>= 10 mm on short axis)",
                      57 :"Chest wall abnormality (bone destruction, metastasis, etc.)",
                      58 :"Consolidation",
                      59 :"Emphysema",
                      60 :"Significant cardiovascular abnormality",
                      61 :"Reticular/reticulonodular opacities, honeycombing, fibrosis, scar",
                      62 :"6 or more nodules, not suspicious for cancer (opacity >= 4 mm)",
                      63 :"Other potentially significant abnormality above the diaphragm",
                      64 :"Other potentially significant abnormality below the diaphragm",
                      65 :"Other minor abnormality noted"}


RACE_DECODER        = { 1:"White",
                        2:"Black or African-American",
                        3:"Asian",
                        4:"American Indian or Alaskan Native",
                        5:"Native Hawaiian or Other Pacific Islander",
                        6:"More than one race",
                        7:"Participant refused to answer",
                        9:"Missing data form - form is not expected to ever be completed",
                        96:"Missing - no response",
                        98:"Missing - form was submitted and the answer was left blank",
                        99:"Unknown/ decline to answer"}




MARGIN_DECODER       = {1 :"Spiculated (Stellate)",
                        2 :"Smooth",
                        3 :"Poorly defined",
                        9 :"Unable to determine"}


ATTN_DECODER        ={1:"Soft Tissue",
                     2:"Ground glass",
                     3:"Mixed",
                     4:"Fluid/water",
                     6:"Fat",
                     7:"Other",
                     9:"Unable to determine"}

LOC_DECODER        ={1:"Right Upper Lobe",
                 2:"Right Middle Lobe",
                 3:"Right Lower Lobe",
                 4:"Left Upper Lobe",
                 5:"Lingula",
                 6:"Left Lower Lobe",
                 8:"Other"}


screening_group_decoder = {
    1: "Screen-detected cancer",
    2: "Has nodule(s)",
    3: "Screened, but no nodules",
    4: "Never screened",
    5: "Other lung cancer"
}


##==== cancer screening 
can_scr_decoder = {
    0: "No Cancer",
    1: "Positive Screen",
    2: "Negative Screen",
    3: "Missed Screen",
    4: "Post Screening"
}

## Start Preprocessing

In [24]:
# decode gender, race, screening group and cancer screening result
all_participants[PATIENT_GENDER_COLMN_NAME]  = all_participants[PATIENT_GENDER_COLMN_NAME].map(GENDER_DECODER)
all_participants[PATIENT_RACE_COLMN_NAME]    = all_participants[PATIENT_RACE_COLMN_NAME].map(RACE_DECODER)
all_participants_ct              = all_participants[all_participants['rndgroup']==1].reset_index(drop=True)
all_participants_ct['scr_group'] = all_participants_ct['scr_group'].map(screening_group_decoder)
all_participants_ct['can_scr']   = all_participants_ct['can_scr'].map(can_scr_decoder)



创建新列primary_cancer_locations，显示每个病人的cancer在哪个位置

In [25]:
location_decoder = {
    '.N': 'Not Applicable',
    0: 'No',
    1: 'Yes'
}

# List of primary cancer location columns
primary_cancer_Loc_columns = ['locunk', 'locoth', 'locmed', 'loccar', 'loclin',
                              'locrhil', 'locrlow', 'locrmid', 'locrmsb', 'locrup',
                              'loclhil', 'locllow', 'loclmsb', 'loclup']

# Convert all location columns to yes/no
for col in primary_cancer_Loc_columns:
    all_participants_ct[col] = all_participants_ct[col].map(location_decoder)

# Select all columns with cancer locations and combine them into a new column
def get_primary_cancer_locations(row):
    locations = [col for col in primary_cancer_Loc_columns if row[col] == 'Yes']
    return ', '.join(locations) if locations else 'No primary location identified'

all_participants_ct['primary_cancer_locations'] = all_participants_ct.apply(get_primary_cancer_locations, axis=1)

构建新cancer lbl：先根据prsn中的'scr_days0','scr_days1','scr_days2','candx_days'构建一个新列screening_year，存储每个patient在第几次screening发现了癌症。之后用ct data构建一个新列，显示这一行是属于第几个screening，最后合并。

In [26]:
screen_day_cols = ['scr_days0', 'scr_days1', 'scr_days2']

# 逻辑：癌症发现日 candx_days 晚于第 x 次 screening 日，则记为 x
# 具体实现为：取所有 <= candx_days 的 screening 中“最后一次”的序号（1/2/3）
def get_cancer_found_after_screening(row):
    candx_day = row['candx_days']
    if pd.isna(candx_day):
        return pd.NA

    screening_days = [row['scr_days0'], row['scr_days1'], row['scr_days2']]
    screening_idx = [i + 1 for i, day in enumerate(screening_days) if pd.notna(day) and candx_day >= day]

    return max(screening_idx) if screening_idx else pd.NA

all_participants_ct['cancer_screening'] = all_participants_ct.apply(
    get_cancer_found_after_screening,
    axis=1
)

读入ctab和ctabc，根据pid，study_yr，sct_ab_desc来进行join

In [27]:
nlst_ctab_idc  = pd.read_csv(read_path+"nlst_780_ctab_idc_20210527.csv")
ctabc = pd.read_csv(read_path+"nlst_780_ctabc_idc_20210527.csv").rename(columns={"sct_ab_code": "sct_ab_desc"})
nlst_ctab_idc = nlst_ctab_idc.merge(
    ctabc,
    on=["pid", "study_yr", "sct_ab_desc"],
    how="left"
)

## 构建新列：表示这次screening发现了多少的nodule
nlst_ctab_idc["num_nodule"] = (
    pd.to_numeric(nlst_ctab_idc["sct_ab_desc"], errors="coerce")
    .isin([51, 52, 53])
    .groupby([nlst_ctab_idc["pid"], nlst_ctab_idc["study_yr"]])
    .transform("sum")
)

nlst_ctab_idc = nlst_ctab_idc[nlst_ctab_idc['sct_ab_desc'] == 51].reset_index(drop=True)
nlst_ctab_idc = nlst_ctab_idc.merge(all_participants_ct, how='left', on='pid')

nlst_ctab_idc["Cancer_lbl"] = (
    (pd.to_numeric(nlst_ctab_idc["study_yr"], errors="coerce") + 1)
    >= pd.to_numeric(nlst_ctab_idc["cancer_screening"], errors="coerce")
).fillna(False).astype("int8")

## age同样需要重大修改

修改age：将scr_days整除365之后加到age之上

In [28]:
def adjust_age_by_study_year(row):
    study_yr = row.get('study_yr')

    if study_yr == 0:
        scr_days = row.get('scr_days0', 0)
    elif study_yr == 1:
        scr_days = row.get('scr_days1', 0)
    elif study_yr == 2:
        scr_days = row.get('scr_days2', 0)
    else:
        scr_days = 0

    if pd.isna(scr_days):
        scr_days = 0

    return row['age'] + (scr_days // 365)

nlst_ctab_idc['age'] = nlst_ctab_idc.apply(adjust_age_by_study_year, axis=1)

## 回到Tushar的处理

In [29]:
# 转换nodule的位置列
nlst_ctab_idc[NODULE_LOC_COLMM_NAME]=nlst_ctab_idc[NODULE_LOC_COLMM_NAME].map(LOC_DECODER)
# 转换nodule的类型列
nlst_ctab_idc[NODULE_ATT_COLMM_NAME]= nlst_ctab_idc[NODULE_ATT_COLMM_NAME].map(ATTN_DECODER)
# 转换nodule的边缘特征列
nlst_ctab_idc[NODULE_MARGIN_COLMM_NAME]=nlst_ctab_idc[NODULE_MARGIN_COLMM_NAME].map(MARGIN_DECODER)


primaimay_loc_DECODER    ={"Right Upper Lobe":'locrup',
                 "Right Middle Lobe":'locrmid',
                 "Right Lower Lobe":'locrlow',
                 "Left Upper Lobe":'loclup',
                 "Lingula": 'loclin',
                 "Left Lower Lobe":'locllow',
                 "Other":'locoth'}    
    
# 构建新列'sct_epi_loc_sform'，将位置映射到和all_participants_ct中primary_cancer_locations列一致的格式
nlst_ctab_idc['sct_epi_loc_sform']=nlst_ctab_idc[NODULE_LOC_COLMM_NAME].map(primaimay_loc_DECODER)

# 若nodule位置在cancer location中，返回1
def match_lobe_mapped(row):
    colmn1_mapped_value = row['sct_epi_loc_sform']  # Assuming this is already mapped
    colmn2_values = row['primary_cancer_locations'].split(',') if pd.notna(row['primary_cancer_locations']) else []
    if pd.isna(colmn1_mapped_value):
        return 0  # No lobe to match
    return 1 if colmn1_mapped_value in colmn2_values else 0

nlst_ctab_idc['match_primary'] = nlst_ctab_idc.apply(match_lobe_mapped, axis=1)


#---- Filtered by non-calcified nodule > 4mm #### 
all_nlst_ctab_idc  = nlst_ctab_idc

##print('Inclusion-exclusion-Crieteria')

print('1) all data |patient:{}|abnornalities entried:{}'.format(len(all_nlst_ctab_idc['pid'].unique()),len(nlst_ctab_idc)))

nlst_ctab_idc      = nlst_ctab_idc[nlst_ctab_idc['sct_ab_desc'].isin([51])].reset_index(drop=True)

print('2) Filtered by non-calcified nodule > 4mm |patient:{}|abnornalities entried:{}'.format(len(nlst_ctab_idc['pid'].unique()),len(nlst_ctab_idc)))

MARGIN_DECODER_Classification       = {"Spiculated (Stellate)":'spiculation',"Smooth":'smooth',"Poorly defined":"poorly-defined","Unable to determine":"poorly-defined"}
ATTN_CLASSIFICATION_MAP             = {"Soft Tissue": "solid","Ground glass": "ground-glass","Mixed": "part-solid","Fluid/water": "part-solid","Fat": "solid","Other": "others","Unable to determine": "others"}

nlst_pancandata                   = nlst_ctab_idc

# 构建新列“家族史”
nlst_pancandata['Family_History'] = nlst_pancandata[['fambrother', 'famchild', 'famfather', 'fammother', 'famsister']].max(axis=1)


##--- Making an clean dataframe from feature importance understanding

columns_for_ml_analysis = ['pid','study_yr','age','gender','all_sct_set','sct_long_dia','sct_margins','sct_pre_att','sct_epi_loc','diagemph','Family_History', 'Cancer_lbl', 'sct_found_after_comp', 'sct_perp_dia','sct_ab_gwth','num_nodule']
                           
nlst_ct_nodule_df       = nlst_pancandata[columns_for_ml_analysis]


#--- Cleaning --

# 去掉NA
nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['sct_long_dia']).reset_index(drop=True)

print('3) Drop missing nodule sizes |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))

nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['sct_pre_att']).reset_index(drop=True)

print('4) Drop missing nodule type  |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))


nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['all_sct_set']).reset_index(drop=True)

print('5) Drop missing Split Set    |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))

nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['Family_History']).reset_index(drop=True)

print('6) Drop missing Family History    |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))


nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['diagemph']).reset_index(drop=True)

print('7) Drop missing Emphysema info    |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))


sct_pre_att_mapping    = {"Soft Tissue": "solid","Ground glass": "ground-glass","Mixed": "part-solid","Fluid/water": "part-solid","Fat": "solid","Other": "others","Unable to determine": "others"}




# 构建新列nodule type，drop掉type为others的行
nlst_ct_nodule_df['Nodule_Type'] = nlst_ct_nodule_df['sct_pre_att'].map(sct_pre_att_mapping)
nlst_ct_nodule_df                = nlst_ct_nodule_df[nlst_ct_nodule_df['Nodule_Type']!='others'].reset_index(drop=True)
print('8) Filtered by Solid, part-solid, ground-glass |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))




# 构建新列Spiculation
sct_margins_mapping               = {"Spiculated (Stellate)":'spiculation',"Smooth":'smooth',"Poorly defined":"poorly-defined","Unable to determine":"poorly-defined"}
nlst_ct_nodule_df['Spiculation']  = nlst_ct_nodule_df['sct_margins'].map(sct_margins_mapping)
nlst_ct_nodule_df['Spiculation']  = nlst_ct_nodule_df['Spiculation'].apply(lambda x: 1 if 'spiculation' in x else 0)



# 根据位置sct_epi_loc'，构建新列'Upper_Lobe'，如果位置在upper或middle lobe，返回1，否则返回0
sct_epi_loc_mapping = {"Right Upper Lobe":'Upper',
                       "Right Middle Lobe":'Upper',
                       "Right Lower Lobe":'Lower',
                       "Left Upper Lobe":'Upper',
                       "Lingula": 'Other',
                       "Left Lower Lobe":'Lower',
                       "Other":'Other'}

nlst_ct_nodule_df['Upper_Lobe']     = nlst_ct_nodule_df['sct_epi_loc'].map(sct_epi_loc_mapping)
nlst_ct_nodule_df['Upper_Lobe']     = nlst_ct_nodule_df['Upper_Lobe'].apply(lambda x: 1 if 'Upper' in x or 'Middle' in x else 0)


1) all data |patient:10257|abnornalities entried:69727
2) Filtered by non-calcified nodule > 4mm |patient:10257|abnornalities entried:69727
3) Drop missing nodule sizes |patient:10239|abnornalities entried:69287
4) Drop missing nodule type  |patient:10238|abnornalities entried:69286
5) Drop missing Split Set    |patient:10204|abnornalities entried:69224
6) Drop missing Family History    |patient:10057|abnornalities entried:68218
7) Drop missing Emphysema info    |patient:10040|abnornalities entried:67986
8) Filtered by Solid, part-solid, ground-glass |patient:9795|abnornalities entried:65512


In [30]:
#---- Set1 and Set2 Division=====#

nlst_ct_nodule_df_set1 = nlst_ct_nodule_df[nlst_ct_nodule_df['all_sct_set']==1].reset_index(drop=True)
nlst_ct_nodule_df_set2 = nlst_ct_nodule_df[nlst_ct_nodule_df['all_sct_set']==2].reset_index(drop=True)


print('Devide by CT Sets')
print('9a) Set-1 |patient:{}|abnornalities entried:{}|Cancer:{}'.format(len(nlst_ct_nodule_df_set1['pid'].unique()),len(nlst_ct_nodule_df_set1),len(nlst_ct_nodule_df_set1[nlst_ct_nodule_df_set1['Cancer_lbl']==1])))
print('9b) Set-2 |patient:{}|abnornalities entried:{}|Cancer:{}'.format(len(nlst_ct_nodule_df_set2['pid'].unique()),len(nlst_ct_nodule_df_set2),len(nlst_ct_nodule_df_set2[nlst_ct_nodule_df_set2['Cancer_lbl']==1])))


save_dir = 'ml_dataset/'

nlst_ct_nodule_df_set1.to_csv(save_dir +'nlst_ct_nodule_df_set1.csv',index=False,encoding='utf-8')
nlst_ct_nodule_df_set2.to_csv(save_dir +'nlst_ct_nodule_df_set2.csv',index=False,encoding='utf-8')

Devide by CT Sets
9a) Set-1 |patient:4896|abnornalities entried:33266|Cancer:1384
9b) Set-2 |patient:4899|abnornalities entried:32246|Cancer:1421
